In [1]:
import sys
sys.path.append("..")
from functools import partial
import pandas as pd
from src.data import load_cuad, TARGET_CLAUSES
from src.split import split_contracts
from src.chunking import clause_chunks
from src.retrieval import make_dense_ranker
from src.bm25 import make_bm25_ranker
from src.hybrid import make_hybrid_ranker
from src.queries import CLAUSE_KEYWORDS
from src.metrics import evaluate_ranker

contracts = load_cuad()
dev_idx, test_idx = split_contracts(contracts)

chunker = partial(clause_chunks, max_size=1500)
tag = "clause-200-1500-200"
dense_q = make_dense_ranker(chunker, tag=tag)
dense_kw = make_dense_ranker(chunker, tag=tag, queries=CLAUSE_KEYWORDS)
bm25 = make_bm25_ranker(chunker)

In [2]:
systems = {
    "dense_question": dense_q,
    "dense_keywords": dense_kw,
    "bm25": bm25,
    "hybrid_kw_1_1": make_hybrid_ranker([dense_kw, bm25]),
}

results = {}
for name, r in systems.items():
    rows = {cl: evaluate_ranker(r, contracts, dev_idx, cl) for cl in TARGET_CLAUSES}
    results[name] = pd.DataFrame(rows).T
    results[name].to_csv(f"../eval/results_{name}_clause1500_dev.csv")
    print("done:", name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

done: dense_question
done: dense_keywords
done: bm25
done: hybrid_kw_1_1


In [3]:
summary = pd.DataFrame(
    {k: v[["hit@1", "hit@3", "mrr"]].mean() for k, v in results.items()}
).T.round(3)
print(summary)
print()
print(pd.DataFrame({k: v["hit@1"] for k, v in results.items()}).round(3))

                hit@1  hit@3    mrr
dense_question  0.611  0.800  0.723
dense_keywords  0.690  0.867  0.788
bm25            0.751  0.891  0.827
hybrid_kw_1_1   0.739  0.908  0.828

                                    dense_question  dense_keywords   bm25  \
Governing Law                                0.683           0.755  0.948   
Renewal Term                                 0.822           0.805  0.890   
Notice Period To Terminate Renewal           0.620           0.634  0.718   
Termination For Convenience                  0.449           0.593  0.788   
Non-Compete                                  0.240           0.507  0.507   
Cap On Liability                             0.700           0.747  0.853   
Uncapped Liability                           0.765           0.790  0.556   

                                    hybrid_kw_1_1  
Governing Law                               0.863  
Renewal Term                                0.856  
Notice Period To Terminate Renewal          0.